In [2]:
import os
import pandas as pd


def convert_and_reindex_parquet_to_jsonl(parquet_path, jsonl_path):
    print(f"🔄 د فایل اړول او بیا رغول پیل شول: {parquet_path}")

    # ډاډ ترلاسه کول چې د محصول فولډر شتون لري
    output_dir = os.path.dirname(jsonl_path)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)

    try:
        # ۱. د Parquet فایل لوستل
        df = pd.read_parquet(parquet_path)
        total_records = len(df)
        print(f"📊 ټول ریکارډونه لوستل شول: {total_records}")

        # ۲. د ډیټا ګډوډول (Shuffle)
        # random_state د دې لپاره دی چې که بیا کوډ چلوئ، د ګډوډۍ پایله کټ مټ یو شان وي (Reproducible)
        df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)
        print("🔀 ډیټا په بریالیتوب سره شفل (Shuffle) شوه.")

        # ۳. د نوې او لنډې آیډي (komak_XXXX) جوړول او مپ کول
        # د 1 څخه پیل کیږي او د ټولو ریکارډونو په اندازه د پډینګ (04d) سره جوړیږي
        new_ids = [f"komak_{i:04d}" for i in range(1, total_records + 1)]
        df_shuffled["id"] = new_ids
        print("🆔 نوې لنډې آیډي ګانې په ترتیب سره ځای پر ځای شوې.")

        # ۴. په JSONL بڼه کې خوندي کول (پښتو نېټیو فونټ ساتلو سره)
        df_shuffled.to_json(
            jsonl_path, orient="records", lines=True, force_ascii=False
        )

        print(f"✅ په بریالیتوب سره بشپړ شو! د محصول فایل: {jsonl_path}")
        print(f"🚀 لومړۍ نمونه: {df_shuffled['id'].iloc[0]} او وروستۍ نمونه: {df_shuffled['id'].iloc[-1]}")

    except Exception as e:
        print(f"❌ تېروتنه رامنځته شوه: {e}")


if __name__ == "__main__":
    # ستاسو د فایلونو ادرسونه
    input_file = "data/train-00000-of-00001.parquet"
    output_file = "data/komak.jsonl"

    convert_and_reindex_parquet_to_jsonl(input_file, output_file)

🔄 د فایل اړول او بیا رغول پیل شول: data/train-00000-of-00001.parquet
📊 ټول ریکارډونه لوستل شول: 3000
🔀 ډیټا په بریالیتوب سره شفل (Shuffle) شوه.
🆔 نوې لنډې آیډي ګانې په ترتیب سره ځای پر ځای شوې.
✅ په بریالیتوب سره بشپړ شو! د محصول فایل: data/komak.jsonl
🚀 لومړۍ نمونه: komak_0001 او وروستۍ نمونه: komak_3000


In [3]:
import json


def print_first_line_jsonl(jsonl_path):
    try:
        with open(jsonl_path, "r", encoding="utf-8") as f:
            # یوازې لومړی لاین لولل
            first_line = f.readline()

            if not first_line:
                print("⚠️ فایل خالي دی!")
                return

            # د سټرینګ اړول په JSON (Python Dictionary) باندې
            data = json.loads(first_line)

            # په زړه پورې او منظم پرنټ کول (indent=2 او ensure_ascii=False د پښتو لپاره)
            print("🔍 د لومړي لاین معلومات په لاندې توګه دي:\n")
            print(json.dumps(data, indent=2, ensure_ascii=False))

    except FileNotFoundError:
        print(f"❌ تېروتنه: په دغه ادرس فایل ونه موندل شو: {jsonl_path}")
    except Exception as e:
        print(f"❌ تېروتنه رامنځته شوه: {e}")


if __name__ == "__main__":
    output_file = "data/komak.jsonl"
    print_first_line_jsonl(output_file)

🔍 د لومړي لاین معلومات په لاندې توګه دي:

{
  "id": "komak_0001",
  "messages": [
    {
      "content": "What are the key concealment strategies employed by individuals or groups when under threat of detection, and how do these strategies adapt based on the nature of the threat and the environment?",
      "role": "user"
    }
  ]
}


In [ ]:
# task is to fix 3000 examples of jsonl lines as {"id":"qa_20250424_065355_653056","messages":[{"content":"What are the crucial considerations when selecting a site for building a natural shelter in a wilderness survival scenario, and what are the three basic types of natural shelters a survivor could construct?","role":"user"}]}
# the ids are too much long we need to make shorter ids like {"id":"komak_0001","messages":[{"content":"What are the crucial considerations when selecting a site for building a natural shelter in a wilderness survival scenario, and what are the three basic types of natural shelters a survivor could construct?","role":"user"}]}
# before making the "id":"komak_0001" shaffle the jsonl lines and then assign new ids in the format "komak_0001", "komak_0002", ..., "komak_3000" to each line.
# د پایتون سکرېپټ (parquet_to_jsonl.py)
import os
import pandas as pd


def convert_parquet_to_jsonl(parquet_path, jsonl_path):
    print(f"🔄 د فایل اړول پیل شول: {parquet_path}")

    # ډاډ ترلاسه کول چې د محصول فولډر شتون لري
    output_dir = os.path.dirname(jsonl_path)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)

    try:
        # د Parquet فایل لوستل
        df = pd.read_parquet(parquet_path)

        # په JSONL بڼه کې خوندي کول (orient='records' او lines=True)
        # د force_ascii=False کارول د دې لپاره دي چې د پښتو توري سم او په نیټیو بڼه خوندي شي
        df.to_json(
            jsonl_path, orient="records", lines=True, force_ascii=False
        )

        print(f"✅ په بریالیتوب سره بشپړ شو! د محصول فایل: {jsonl_path}")
        print(f"📊 ټول ریکارډونه: {len(df)}")

    except Exception as e:
        print(f"❌ تېروتنه رامنځته شوه: {e}")


if __name__ == "__main__":
    # ستاسو د فایلونو ادرسونه
    input_file = "data/train-00000-of-00001.parquet"
    output_file = "data/train.jsonl"

    convert_parquet_to_jsonl(input_file, output_file)

🔄 د فایل اړول پیل شول: data/train-00000-of-00001.parquet
✅ په بریالیتوب سره بشپړ شو! د محصول فایل: data/train.jsonl
📊 ټول ریکارډونه: 3000
